# Experiment 5: Shallow and Deep Parsing
**Mini-project:** Resume Analyzer Using Natural Language Processing (Group 28)

**Objective:** Implement POS tagging, shallow parsing (chunking), deep (dependency) parsing and CFG-based parse tree generation on resume sentences.

Open in Colab: [Exp5_Parsing.ipynb](https://colab.research.google.com/github/GnsShubham06/resume-analyzer-nlp/blob/main/notebooks/experiments/Exp5_Parsing.ipynb) &nbsp;|&nbsp; Run with **Runtime → Run all**.
Data: 100 resumes (20 per domain); the screenshots in the report use five fictional (synthetic) resumes.

In [1]:
# ---- Setup: runs in Google Colab and locally --------------------------------
# In Colab this cell downloads the project repository (code + anonymised
# dataset) from GitHub and installs the packages Colab does not ship with.
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
REPO = "https://github.com/GnsShubham06/resume-analyzer-nlp"
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/resume-analyzer-nlp"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/content/resume-analyzer-nlp"], check=True)
    os.chdir("/content/resume-analyzer-nlp")
    extra = []
    if extra:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
else:  # local run from notebooks/experiments/
    for _ in range(3):
        if os.path.isfile("requirements.txt"):
            break
        os.chdir("..")
sys.path.insert(0, os.getcwd())

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'universal_tagset']:
    nltk.download(pkg, quiet=True)

import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:  # model not installed yet (fresh Colab runtime)
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)
    import importlib, site
    importlib.invalidate_caches(); importlib.reload(site)
    import en_core_web_sm
    nlp = en_core_web_sm.load()

%config InlineBackend.figure_format = "retina"
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.read_csv("data/resume_dataset.csv")
df["extracted_text"] = df["extracted_text"].fillna("")
print("Project folder:", os.path.basename(os.getcwd()), "| resumes:", len(df))

Project folder: resume-analyzer-nlp | resumes: 100


In [2]:
from nltk.tag import map_tag
s1 = "Prepared monthly reports for the leadership team."                         # Finance bullet
s2 = "Managed seed production plots and recommended pest control measures."     # Agriculture bullet
UNI = {".": "PUNCT", "CONJ": "CCONJ", "PRT": "PART"}
def nltk_tags(sentence, bullet_fix=False):
    words = nltk.word_tokenize(sentence)
    if bullet_fix:   # resume bullets have no subject: tag "I <verb> ..." and drop the "I"
        return nltk.pos_tag(["I"] + [words[0].lower()] + words[1:])[1:]
    return nltk.pos_tag(words)
def universal(tag):
    u = map_tag("en-ptb", "universal", tag)
    return UNI.get(u, u)

## a. POS tagging (NLTK vs spaCy)

In [3]:
def compare(sentence):
    raw, fixed = nltk_tags(sentence), nltk_tags(sentence, bullet_fix=True)
    sp = [t.pos_ for t in nlp(sentence)]
    uni = [universal(t) for _, t in fixed]
    return pd.DataFrame([[w for w, _ in raw], [t for _, t in raw], [t for _, t in fixed], uni, sp,
                         ["yes" if a == b else "no" for a, b in zip(uni, sp)]],
                        index=["word", "NLTK raw", "NLTK bullet-fix", "NLTK universal", "spaCy", "agree"])
t1, t2 = compare(s1), compare(s2)
display(t1)
first_raw = sum(nltk_tags(s)[0][1].startswith("VB") for s in (s1, s2))
first_fix = sum(nltk_tags(s, True)[0][1].startswith("VB") for s in (s1, s2))
agree = pd.concat([t1.loc["agree"], t2.loc["agree"]])
print(f"first word tagged as verb: raw NLTK {first_raw}/2, with bullet fix {first_fix}/2")
print(f"agreement (bullet-fix NLTK, universal tags) vs spaCy: {(agree == 'yes').mean():.0%} of {len(agree)} tokens")

,0,1,2,3,4,5,6,7
word,Prepared,monthly,reports,for,the,leadership,team,.
NLTK raw,NNP,JJ,NNS,IN,DT,NN,NN,.
NLTK bullet-fix,VBD,JJ,NNS,IN,DT,NN,NN,.
NLTK universal,VERB,ADJ,NOUN,ADP,DET,NOUN,NOUN,PUNCT
spaCy,VERB,ADJ,NOUN,ADP,DET,NOUN,NOUN,PUNCT
agree,yes,yes,yes,yes,yes,yes,yes,yes


first word tagged as verb: raw NLTK 0/2, with bullet fix 2/2
agreement (bullet-fix NLTK, universal tags) vs spaCy: 100% of 18 tokens


## b. Shallow parsing / chunking

In [4]:
from nltk.chunk import tree2conlltags
grammar = r"""
  NP: {<DT>?<JJ.*>*<NN.*>+}
  PP: {<IN><NP>}
  VP: {<VB.*><NP|PP>+}
"""
chunker = nltk.RegexpParser(grammar)
tagged1 = [(w.lower(), t) for w, t in nltk_tags(s1, bullet_fix=True)]
tree = chunker.parse(tagged1)
print(tree)
print("NP chunks:", [" ".join(w for w, _ in st.leaves()) for st in tree.subtrees(lambda t: t.label() == "NP")])
np_chunker = nltk.RegexpParser(r"NP: {<DT>?<JJ.*>*<NN.*>+}")
tagged2 = [(w.lower(), t) for w, t in nltk_tags(s2, bullet_fix=True)]
print("IOB tags :", tree2conlltags(np_chunker.parse(tagged2)))

(S
  (VP
    prepared/VBD
    (NP monthly/JJ reports/NNS)
    (PP for/IN (NP the/DT leadership/NN team/NN)))
  ./.)
NP chunks: ['monthly reports', 'the leadership team']
IOB tags : [('managed', 'VBD', 'O'), ('seed', 'NN', 'B-NP'), ('production', 'NN', 'I-NP'), ('plots', 'NNS', 'I-NP'), ('and', 'CC', 'O'), ('recommended', 'VBD', 'O'), ('pest', 'JJS', 'B-NP'), ('control', 'NN', 'I-NP'), ('measures', 'NNS', 'I-NP'), ('.', '.', 'O')]


## c. Deep parsing – dependency parse

In [5]:
from spacy import displacy
from IPython.display import HTML
doc = nlp("I " + s2[0].lower() + s2[1:])
for t in doc:
    print(f"{t.text:<14}--{t.dep_:>8}--> {t.head.text}")
display(HTML(displacy.render(doc, style="dep", jupyter=False,
                             options={"compact": True, "distance": 100, "color": "#000000"})))

I             --   nsubj--> managed
managed       --    ROOT--> managed
seed          --compound--> plots
production    --compound--> plots
plots         --    dobj--> managed
and           --      cc--> managed
recommended   --    conj--> managed
pest          --    amod--> measures
control       --compound--> measures
measures      --    dobj--> recommended
.             --   punct--> managed


## d. CFG-based parse tree generation

In [6]:
cfg = nltk.CFG.fromstring("""
  S   -> VP
  VP  -> V NP | V NP PP
  NP  -> ADJ N | DET ADJ N | NP PP
  PP  -> P NP
  V   -> 'prepared'
  ADJ -> 'monthly' | 'leadership'
  N   -> 'reports' | 'team'
  P   -> 'for'
  DET -> 'the'
""")
sent = "prepared monthly reports for the leadership team".split()
trees = list(nltk.ChartParser(cfg).parse(sent))
trees.sort(key=lambda t: t[0][1].label() == "NP" and len(t[0]) == 2)   # verb attachment first
print("sentence:", sent)
print(f"number of parse trees: {len(trees)} (PP-attachment ambiguity)")
for i, t in enumerate(trees, 1):
    where = "VERB" if len(t[0]) == 3 else "NOUN"
    print(f"--- tree {i}: 'for the leadership team' attaches to the {where}")
    t.pretty_print()

sentence: ['prepared', 'monthly', 'reports', 'for', 'the', 'leadership', 'team']
number of parse trees: 2 (PP-attachment ambiguity)
--- tree 1: 'for the leadership team' attaches to the VERB
                              S                         
                              |                          
                              VP                        
    __________________________|_______                   
   |              |                   PP                
   |              |            _______|______            
   |              NP          |              NP         
   |         _____|_____      |    __________|_______    
   V       ADJ          N     P  DET        ADJ      N  
   |        |           |     |   |          |       |   
prepared monthly     reports for the     leadership team

--- tree 2: 'for the leadership team' attaches to the NOUN
            S                                           
            |                                            
  